# Sorting Notebook

This notebook will download and sort electrophysiology collected using an Intan headstage, in the .rhd format. 

The data is intracranial mouse recording, from a 16 channel microarray. The paper can be found here: https://doi.org/10.1371/journal.pone.0221510


# Getting Set Up

1. Open a terminal. Make sure "Sorter" environment is active by running these commands in the terminal 

```
conda deactivate
conda activate sorter
```


In [ ]:
import sys
import torch
import sklearn

from pathlib import Path
import spikeinterface as si

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: /home/dhn/Documents/development/zlab_sorting_script/.venv/bin/python
PyTorch: 2.12.0+cu130
CUDA available: True
GPU: NVIDIA T400 4GB


In [2]:
from sorting_scripts import zsort, auto_curate

In [3]:
patient = "DB007"
session = "Task_261019_165842"

path_dict = zsort.set_paths(
    "/mnt/MED_Data_iSSD/Neural_Data/DB007/Task_261019_165842"
)

rec = zsort.load_recording(path_dict)

rec = zsort.set_probe(
    rec, path_dict,
    "deeparray128_P1925-127_H1925-116.json",
)

[zsort] Loaded 15 RHD file(s): 128 channels, 30000 Hz, 25435904 samples, 847.863 seconds.
no probe, attaching one


# Load sorter and analyzer if they exist

In [ ]:
# Paths
SORTER_PATH = path_dict["sorter_output_folder"]
ANALYZER_PATH = path_dict["analyzer_folder"]

# Load sorting
sort = si.sorters.read_sorter_folder(SORTER_PATH)

# Load analyzer
analyzer = si.load_sorting_analyzer(ANALYZER_PATH)

KeyError: '/mnt/MED_Data_iSSD/Neural_Data/DB007/Task_261019_165842/DB007-Task_261019_165842-sorted/DB007-Task_261019_165842-sorter_folder'

# If they do not exist, run the sorter and create analyzer

In [ ]:
retry_paths = dict(path_dict)
for key in ("sorter_output_folder", "analyzer_folder"):
    old = Path(path_dict[key])
    retry_paths[key] = old.with_name(old.name + "_lowmem")

sort = zsort.sort(
    rec, retry_paths,
    batch_size=15000,
    clear_cache=True,
)
analyzer = zsort.analyze(rec, sort, retry_paths)

[zsort] recording dtype is unsigned (uint16); converting to signed


write_binary_recording (no parallelization):   0%|          | 0/848 [00:00<?, ?it/s]

In [ ]:
# 6. Sort
sort = zsort.sort(rec, path_dict,
                  batch_size=15000,
                  clear_cache=True)



[zsort] recording dtype is unsigned (uint16); converting to signed


write_binary_recording (no parallelization):   0%|          | 0/848 [00:00<?, ?it/s]

kilosort.run_kilosort:  
kilosort.run_kilosort: Computing preprocessing variables.
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: N samples: 25435904
kilosort.run_kilosort: N seconds: 847.8634666666667
kilosort.run_kilosort: N batches: 1696
kilosort.run_kilosort: Preprocessing filters computed in 1.64s; total 1.64s
kilosort.run_kilosort:  
kilosort.run_kilosort: Resource usage after preprocessing
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort: CPU usage:     4.10 %
kilosort.run_kilosort: Mem used:     20.10 %     |      12.56 GB
kilosort.run_kilosort: Mem avail:    49.94 / 62.50 GB
kilosort.run_kilosort: ------------------------------------------------------
kilosort.run_kilosort: GPU usage:    `conda install pynvml` for GPU usage
kilosort.run_kilosort: GPU memory:   30.49 %     |      1.10   /     3.62 GB
kilosort.run_kilosort: Allocated:     0.22 %     |      0.01   /     3.62 GB
kilosort.r

kilosort4 run time 5106.60s
[zsort] recording dtype is unsigned (uint16); converting to signed


estimate_sparsity (workers: 32 processes):   0%|          | 0/848 [00:00<?, ?it/s]

...

noise_level (workers: 20 processes):   0%|          | 0/20 [00:00<?, ?it/s]

/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/spikeinterface/postprocessing/localization_tools.py:341: RuntimeWarning: invalid value encountered in divide
  com = np.sum(wf_data[:, np.newaxis] * local_contact_locations, axis=0) / np.sum(wf_data)
/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/spikeinterface/postprocessing/localization_tools.py:367: UserWarning: scipy.optimize.least_squares error: Each lower bound must be strictly less than each upper bound.
  warnings.warn(f"scipy.optimize.least_squares error: {e}")


Fitting PCA:   0%|          | 0/320 [00:00<?, ?it/s]

Projecting waveforms:   0%|          | 0/320 [00:00<?, ?it/s]

Compute : spike_amplitudes (workers: 32 processes):   0%|          | 0/848 [00:00<?, ?it/s]

In [9]:
# 7. Analyze
analyzer = zsort.analyze(rec, sort, path_dict)

NameError: name 'sort' is not defined

In [11]:
import spikeinterface as si

curated_data = si.load_sorting_analyzer(
    "/mnt/MED_Data_iSSD/Neural_Data/DB007/Task_261019_165842/"
    "DB007-Task_261019_165842-sorted/"
    "auto_curated_20261023T232213_d5608e15/analyzer"
)

In [12]:
analyzer = si.load_sorting_analyzer(path_dict["analyzer_folder"])

/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/sklearn/base.py:376: InconsistentVersionWarning: Trying to unpickle estimator IncrementalPCA from version 1.9.0 when using version 1.4.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## Automated curation and ML review labels

Cleanup and stock hard merges run first. UnitRefine then labels the final unit IDs:
noise/neural for all units, followed by SUA/MUA for neural predictions only.
Predictions are advisory: the configuration below keeps all post-cleanup units.
Threshold quality labels are still computed, but neither they nor ML predictions
remove units unless their separate filtering switches are enabled.

The models were trained on mouse Neuropixels, **not validated for human DeepArray**.
`model_trust=True` allows loading the requested HF model artifacts; only use trusted
repositories. Missing cached models download automatically. Load/inference errors
stop the run and leave a diagnostic report. The current models need scikit-learn 1.4.2,
but this environment has 1.9.0: use a separate compatible environment or temporarily
set `enable_model_labeling=False`. No installed packages are changed automatically.
Training metric parameters are absent
from the current models; known settings are enforced, unknown settings are recorded.

All orchestration lives in `src`; this notebook contains only policy and review calls.
See [the workflow guide](../docs/auto_curation.md) and
[the implementation audit](../docs/model_labeling_audit.md).


In [5]:
curation_config = auto_curate.AutoCurationConfig(
    label_method="threshold",
    thresholds={
        "snr": {"greater": 5.0},
        "firing_rate": {"greater": 0.1},
        "isi_violations_ratio": {"less": 0.5},
    },
    apply_quality_filtering=False,
    enable_model_labeling=True,
    enable_noise_classifier=True,
    enable_sua_classifier=True,
    models={
        "noise": "SpikeInterface/UnitRefine_noise_neural_classifier",
        "sua_mua": "SpikeInterface/UnitRefine_sua_mua_classifier",
    },
    model_trust=True,
    enforce_metric_params=True,
    apply_model_filtering=False,
    model_filtering={},
    remove_duplicate_spikes=True,
    remove_redundant_units=True,
    auto_merge=True,
    merge_preset="similarity_correlograms",
    export_report=True,
    export_phy=False,
    job_kwargs={"n_jobs": 1, "chunk_duration": "1s", "progress_bar": True},
)

In [ ]:
from sorting_scripts import auto_curate
auto_curate.preflight(curation_config) 

1.4.2


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

{'status': 'passed',
 'models': {'noise': {'repo_id': 'SpikeInterface/UnitRefine_noise_neural_classifier',
   'revision': '280cd07ff8310286ded5c0cfdb382e92e2a43a98',
   'sha256': 'f946d77324393959a49e2b83df2083326ea5a934e068eb48f2a362a11e551067',
   'requirements': {'spikeinterface': '0.102.0', 'scikit-learn': '1.4.2'},
   'required_metrics': ['amplitude_cutoff',
    'amplitude_cv_median',
    'amplitude_cv_range',
    'amplitude_median',
    'drift_ptp',
    'drift_std',
    'drift_mad',
    'firing_range',
    'firing_rate',
    'isi_violations_ratio',
    'isi_violations_count',
    'num_spikes',
    'presence_ratio',
    'rp_contamination',
    'rp_violations',
    'sliding_rp_violation',
    'snr',
    'sync_spike_2',
    'sync_spike_4',
    'sync_spike_8',
    'd_prime',
    'isolation_distance',
    'l_ratio',
    'silhouette',
    'nn_hit_rate',
    'nn_miss_rate',
    'exp_decay',
    'half_width',
    'num_negative_peaks',
    'num_positive_peaks',
    'peak_to_valley',
    '

In [7]:
curation_result = auto_curate.auto_curate(analyzer, path_dict, curation_config)
curated_data = curation_result.analyzer
curation_result.summary()

/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/spikeinterface/core/baserecordingsnippets.py:242: UserWarning: There is no Probe attached to this recording. Creating a dummy one with contact positions
  warn("There is no Probe attached to this recording. Creating a dummy one with contact positions")


compute_waveforms (no parallelization):   0%|          | 0/848 [00:00<?, ?it/s]

/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/spikeinterface/core/recording_tools.py:780: UserWarning: get_noise_levels(recording, num_chunks_per_segment=20) is deprecated
Now, you need to use get_noise_levels(recording, random_slices_kwargs=dict(num_chunks_per_segment=20, chunk_size=1000))
Please read get_random_recording_slices() documentation for more options.
  warnings.warn(msg)


noise_level (no parallelization):   0%|          | 0/20 [00:00<?, ?it/s]

/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/spikeinterface/postprocessing/localization_tools.py:341: RuntimeWarning: invalid value encountered in divide
  com = np.sum(wf_data[:, np.newaxis] * local_contact_locations, axis=0) / np.sum(wf_data)
/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/spikeinterface/postprocessing/localization_tools.py:367: UserWarning: scipy.optimize.least_squares error: Each lower bound must be strictly less than each upper bound.
  warnings.warn(f"scipy.optimize.least_squares error: {e}")


...

spike_locations (no parallelization):   0%|          | 0/848 [00:00<?, ?it/s]

/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/spikeinterface/qualitymetrics/misc_metrics.py:1066: UserWarning: Some units have too few spikes : amplitude_cutoff is set to NaN
  warnings.warn(f"Some units have too few spikes : amplitude_cutoff is set to NaN")
/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/numpy/core/_methods.py:163: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


calculate pc_metrics:   0%|          | 0/312 [00:00<?, ?it/s]

/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/spikeinterface/postprocessing/template_metrics.py:303: UserWarning: With less than 10 channels, multi-channel metrics might not be reliable.
  warnings.warn(
/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/spikeinterface/postprocessing/template_metrics.py:303: UserWarning: With less than 10 channels, multi-channel metrics might not be reliable.
  warnings.warn(
/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/spikeinterface/postprocessing/template_metrics.py:303: UserWarning: With less than 10 channels, multi-channel metrics might not be reliable.
  warnings.warn(
/home/dhn/Documents/development/zlab_sorting_script/.venv/lib/python3.12/site-packages/spikeinterface/postprocessing/template_metrics.py:303: UserWarning: With less than 10 channels, multi-channel metrics might not be reliable.
  warnings.warn(
/home/dhn/Documents/

{'input_units': 320,
 'kept_units': 312,
 'merged_groups': 2,
 'original_unit_decisions': {'kept': 310, 'redundant': 6, 'merged_kept': 4},
 'output_folder': '/mnt/MED_Data_iSSD/Neural_Data/DB007/Task_261019_165842/DB007-Task_261019_165842-sorted/auto_curated_20261023T232213_d5608e15'}

In [13]:
display(curation_result.ml_classifications)
display(curation_result.decisions)
display(curation_result.labels)
print("Saved analyzer:", curation_result.analyzer_folder)
print("ML report and model metadata:", curation_result.output_folder)

,noise_label,noise_probability,noise_status,noise_model_id,noise_model_revision,sua_mua_label,sua_mua_probability,sua_mua_status,sua_mua_model_id,sua_mua_model_revision,model_labeling_status,filter_candidate,filter_applied,review_required,retained
unit_id,,,,,,,,,,,,,,,
0,noise,0.949463,evaluated,SpikeInterface/UnitRefine_noise_neural_classifier,280cd07ff8310286ded5c0cfdb382e92e2a43a98,,NaN,not_evaluated,SpikeInterface/UnitRefine_sua_mua_classifier,9ee216e7e0e7dbf0778cb07a8d1753700dbbad3b,evaluated,False,False,True,True
1,noise,0.919335,evaluated,SpikeInterface/UnitRefine_noise_neural_classifier,280cd07ff8310286ded5c0cfdb382e92e2a43a98,,NaN,not_evaluated,SpikeInterface/UnitRefine_sua_mua_classifier,9ee216e7e0e7dbf0778cb07a8d1753700dbbad3b,evaluated,False,False,True,True
2,noise,0.954497,evaluated,SpikeInterface/UnitRefine_noise_neural_classifier,280cd07ff8310286ded5c0cfdb382e92e2a43a98,,NaN,not_evaluated,SpikeInterface/UnitRefine_sua_mua_classifier,9ee216e7e0e7dbf0778cb07a8d1753700dbbad3b,evaluated,False,False,True,True
4,noise,0.746618,evaluated,SpikeInterface/UnitRefine_noise_neural_classifier,280cd07ff8310286ded5c0cfdb382e92e2a43a98,,NaN,not_evaluated,SpikeInterface/UnitRefine_sua_mua_classifier,9ee216e7e0e7dbf0778cb07a8d1753700dbbad3b,evaluated,False,False,True,True
5,noise,0.903454,evaluated,SpikeInterface/UnitRefine_noise_neural_classifier,280cd07ff8310286ded5c0cfdb382e92e2a43a98,,NaN,not_evaluated,SpikeInterface/UnitRefine_sua_mua_classifier,9ee216e7e0e7dbf0778cb07a8d1753700dbbad3b,evaluated,False,False,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
317,neural,0.609939,evaluated,SpikeInterface/UnitRefine_noise_neural_classifier,280cd07ff8310286ded5c0cfdb382e92e2a43a98,mua,0.719021,evaluated,SpikeInterface/UnitRefine_sua_mua_classifier,9ee216e7e0e7dbf0778cb07a8d1753700dbbad3b,evaluated,False,False,True,True
318,neural,0.759997,evaluated,SpikeInterface/UnitRefine_noise_neural_classifier,280cd07ff8310286ded5c0cfdb382e92e2a43a98,sua,0.699115,evaluated,SpikeInterface/UnitRefine_sua_mua_classifier,9ee216e7e0e7dbf0778cb07a8d1753700dbbad3b,evaluated,False,False,True,True
319,neural,0.606732,evaluated,SpikeInterface/UnitRefine_noise_neural_classifier,280cd07ff8310286ded5c0cfdb382e92e2a43a98,mua,0.625576,evaluated,SpikeInterface/UnitRefine_sua_mua_classifier,9ee216e7e0e7dbf0778cb07a8d1753700dbbad3b,evaluated,False,False,True,True


,decision,final_unit_id,label
unit_id,,,
0,kept,0,noise
1,kept,1,noise
2,kept,2,noise
3,redundant,NaN,
4,kept,4,noise
...,...,...,...
315,kept,315,noise
316,kept,316,noise
317,kept,317,noise


,label,reason
0,noise,snr;isi_violations_ratio
1,noise,snr;isi_violations_ratio
2,noise,snr;isi_violations_ratio
4,noise,snr;firing_rate
5,noise,snr
...,...,...
317,noise,snr
318,good,
319,noise,snr
320,noise,snr


Saved analyzer: /mnt/MED_Data_iSSD/Neural_Data/DB007/Task_261019_165842/DB007-Task_261019_165842-sorted/auto_curated_20261023T232213_d5608e15/analyzer
ML report and model metadata: /mnt/MED_Data_iSSD/Neural_Data/DB007/Task_261019_165842/DB007-Task_261019_165842-sorted/auto_curated_20261023T232213_d5608e15


### Visual review

The Unit List displays separate ML labels, numeric confidence, and evaluation status.
Noise-predicted units have no fabricated SUA/MUA label. ML fields never replace human
curation labels. The companion `unit_ml_classifications.csv` uses the same final IDs
as the GUI; `decisions.csv` maps them back to source units. Each run saves independently.
Opening this viewer is read-only; no GUI Save action is needed for automated output.


In [14]:
window = zsort.launch_curation_gui(
    curated_data, curation=False, **curation_result.gui_kwargs()
)

Found available port: 57133
Launching server at http://localhost:57133


In [ ]:
zsort.generate_figures(curated_data, path_dict)

In [17]:
# # Interact with box via api

# !box folders:items 352606395707

In [18]:
# !box folders:items 352606396623

In [19]:
# !box folders:items 352605477299

In [20]:
# !box folders:upload 354623627238

In [21]:
# !box folders:upload "/home/marco/codespace/data/Intan_RDH_2000/Session1/sorted/cleaned_analyzer.zarr" -p 354623627238